In [1]:
import torch
from torch import nn
import torchvision
from torchvision import transforms
from torch.utils.tensorboard import SummaryWriter # <-- Ajout pour TensorBoard
from tqdm import tqdm  # Version standard pour éviter les bugs d'affichage

# Sélection automatique de l'accélérateur
device = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps") if torch.backends.mps.is_available() else
    torch.device("cpu")
)

print("Device utilisé :", device)

Device utilisé : cuda


In [2]:
# Valeurs de normalisation spécifiques à CIFAR-10
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2023, 0.1994, 0.2010)

# Pipeline Train : Augmentation + Conversion Tensor + Normalisation
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4), # Augmentation supplémentaire courante
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar10_mean, std=cifar10_std) # <-- Normalisation
])

# Pipeline Test : Conversion Tensor + Normalisation (Pas d'augmentation sur le test !)
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar10_mean, std=cifar10_std) # <-- Normalisation
])

# Téléchargement et DataLoaders
dataset_train = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform_train, download=True)
dataset_test = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform_test, download=True)

dataloader_train = torch.utils.data.DataLoader(dataset_train, batch_size=64, shuffle=True)
dataloader_test = torch.utils.data.DataLoader(dataset_test, batch_size=64, shuffle=False)

In [3]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Bloc d'extraction des caractéristiques (Convolutions)
        self.features = nn.Sequential(
            # Bloc 1 : Entrée (3, 32, 32) -> Sortie (32, 16, 16) après MaxPool
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Bloc 2 : Entrée (32, 16, 16) -> Sortie (64, 8, 8) après MaxPool
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25)
        )
        
        # Bloc de classification (Couches denses)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10)  # 10 classes de sortie
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [4]:
# Initialisation de l'outil d'enregistrement TensorBoard
writer = SummaryWriter("runs/cifar10_cnn_experiment")

# Instanciation du modèle (garde la classe CNN définie précédemment)
cnn_model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=1e-3)

# Optionnel : On envoie la structure du graphe de ton modèle à TensorBoard
images, labels = next(iter(dataloader_train))
writer.add_graph(cnn_model, images.to(device))

In [5]:
def train_loop(dataloader, model, loss_fn, optimizer, epoch, writer):
    model.train()
    progress_bar = tqdm(dataloader, desc=f"Époque {epoch} [Train]", leave=False)
    
    for batch, (X, y) in enumerate(progress_bar):
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Calcul du pas global pour TensorBoard
        global_step = epoch * len(dataloader) + batch
        
        # Enregistrement de la Loss dans TensorBoard
        writer.add_scalar("Loss/Train_Step", loss.item(), global_step)

        if batch % 10 == 0:
            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

In [6]:
def test_loop(dataloader, model, loss_fn, epoch, writer):
    model.eval()
    size = len(dataloader.dataset)
    correct = 0
    total_loss = 0

    progress_bar = tqdm(dataloader, desc=f"Époque {epoch} [Eval]", leave=False)

    with torch.no_grad():
        for X, y in progress_bar:
            X, y = X.to(device), y.to(device)
            pred = model(X)

            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()

    accuracy = 100 * correct / size
    avg_loss = total_loss / len(dataloader)

    # Enregistrement des métriques par Époque dans TensorBoard
    writer.add_scalar("Loss/Test_Epoch", avg_loss, epoch)
    writer.add_scalar("Accuracy/Test_Epoch", accuracy, epoch)

    print(f"--> Époque {epoch} : Accuracy: {accuracy:.1f}%, Avg loss: {avg_loss:.4f}")

In [9]:
def run_training():
    epochs = 10 # Augmenté à 10 pour mieux apprécier les courbes TensorBoard
    for t in range(epochs):
        # On passe 't' (l'époque actuelle) et le 'writer' aux fonctions
        train_loop(dataloader_train, cnn_model, criterion, optimizer, t, writer)
        test_loop(dataloader_test, cnn_model, criterion, t, writer)
    
    # Très important : on ferme le writer pour s'assurer que tout est écrit sur le disque
    writer.close()
    print("\nEntraînement terminé ! Courbes prêtes sur TensorBoard ")

run_training()

--> Époque 0 : Accuracy: 81.3%, Avg loss: 0.5476


--> Époque 1 : Accuracy: 80.8%, Avg loss: 0.5649


--> Époque 2 : Accuracy: 80.8%, Avg loss: 0.5660


--> Époque 3 : Accuracy: 81.7%, Avg loss: 0.5482


--> Époque 4 : Accuracy: 80.6%, Avg loss: 0.5636


--> Époque 5 : Accuracy: 81.8%, Avg loss: 0.5335


--> Époque 6 : Accuracy: 82.0%, Avg loss: 0.5256


--> Époque 7 : Accuracy: 81.8%, Avg loss: 0.5380


--> Époque 8 : Accuracy: 81.8%, Avg loss: 0.5324


--> Époque 9 : Accuracy: 81.4%, Avg loss: 0.5491

Entraînement terminé ! Courbes prêtes sur TensorBoard 
